<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  Copyright (c) Microsoft Corporation.
#  Licensed under the MIT License.

In [1]:
import sys, site
from pathlib import Path

################################# NOTE #################################
#  Please be aware that if colab installs the latest numpy and pyqlib  #
#  in this cell, users should RESTART the runtime in order to run the  #
#  following cells successfully.                                       #
########################################################################

try:
    import qlib
except ImportError:
    # install qlib
    ! pip install --upgrade numpy
    ! pip install pyqlib
    if "google.colab" in sys.modules:
        # The Google colab environment is a little outdated. We have to downgrade the pyyaml to make it compatible with other packages
        ! pip install pyyaml==5.4.1
    # reload
    site.main()

scripts_dir = Path.cwd().parent.joinpath("scripts")
if not scripts_dir.joinpath("get_data.py").exists():
    # download get_data.py script
    scripts_dir = Path("~/tmp/qlib_code/scripts").expanduser().resolve()
    scripts_dir.mkdir(parents=True, exist_ok=True)
    import requests

    with requests.get("https://raw.githubusercontent.com/microsoft/qlib/main/scripts/get_data.py", timeout=10) as resp:
        with open(scripts_dir.joinpath("get_data.py"), "wb") as fp:
            fp.write(resp.content)

In [2]:
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# use default data
# NOTE: need to download data from remote: python scripts/get_data.py qlib_data_cn --target_dir ~/.qlib/qlib_data/cn_data
provider_uri = "~/.qlib/qlib_data/cn_data"  # target_dir
if not exists_qlib_data(provider_uri):
    print(f"Qlib data is not found in {provider_uri}")
    sys.path.append(str(scripts_dir))
    from get_data import GetData

    GetData().qlib_data(target_dir=provider_uri, region=REG_CN)
qlib.init(provider_uri=provider_uri, region=REG_CN)

[84078:MainThread](2026-07-16 19:21:49,122) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[84078:MainThread](2026-07-16 19:21:49,125) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[84078:MainThread](2026-07-16 19:21:49,125) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/xuxin/.qlib/qlib_data/cn_data')}


In [4]:
market = "csi300"
benchmark = "SH000300"

# train model

In [5]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2020-08-01",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": market,
}

task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2014-12-31"),
                "valid": ("2015-01-01", "2016-12-31"),
                "test": ("2017-01-01", "2020-08-01"),
            },
        },
    },
}

# model initialization
model = init_instance_by_config(task["model"])
dataset = init_instance_by_config(task["dataset"])

# start exp to train model
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

[84078:MainThread](2026-07-16 19:23:11,613) INFO - qlib.timer - [log.py:127] - Time cost: 68.410s | Loading data Done
[84078:MainThread](2026-07-16 19:23:11,849) INFO - qlib.timer - [log.py:127] - Time cost: 0.106s | DropnaLabel Done
[84078:MainThread](2026-07-16 19:23:12,755) INFO - qlib.timer - [log.py:127] - Time cost: 0.905s | CSZScoreNorm Done
[84078:MainThread](2026-07-16 19:23:12,760) INFO - qlib.timer - [log.py:127] - Time cost: 1.145s | fit & process data Done
[84078:MainThread](2026-07-16 19:23:12,761) INFO - qlib.timer - [log.py:127] - Time cost: 69.560s | Init data Done
[84078:MainThread](2026-07-16 19:23:12,774) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name train_model.
[84078:MainThread](2026-07-16 19:23:12,787) INFO - qlib.workflow - [exp.py:258] - Experiment 422192818557302453 starts running ...
[84078:MainThread](2026-07-16 19:23:13,039) INFO - qlib.workflow - [recorder.py:345] - Recorder 34ce8186e2e246f88a588145

Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.990585	valid's l2: 0.99431
[40]	train's l2: 0.986931	valid's l2: 0.993693
[60]	train's l2: 0.984352	valid's l2: 0.99349
[80]	train's l2: 0.982278	valid's l2: 0.993398
[100]	train's l2: 0.980459	valid's l2: 0.993321
[120]	train's l2: 0.978684	valid's l2: 0.993361
[140]	train's l2: 0.976979	valid's l2: 0.993392
Early stopping, best iteration is:
[100]	train's l2: 0.980459	valid's l2: 0.993321


[84078:MainThread](2026-07-16 19:23:26,596) INFO - qlib.timer - [log.py:127] - Time cost: 0.243s | waiting `async_log` Done


# prediction, backtest & analysis

In [6]:
###################################
# prediction, backtest & analysis
###################################
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "model": model,
            "dataset": dataset,
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": "2017-01-01",
        "end_time": "2020-08-01",
        "account": 100000000,
        "benchmark": benchmark,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5,
        },
    },
}

# backtest and analysis
with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model")

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

[84078:MainThread](2026-07-16 19:23:38,908) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name backtest_analysis.
[84078:MainThread](2026-07-16 19:23:38,920) INFO - qlib.workflow - [exp.py:258] - Experiment 892219596593616876 starts running ...
[84078:MainThread](2026-07-16 19:23:38,970) INFO - qlib.workflow - [recorder.py:345] - Recorder cbe27a3047a9456b8bb4980b64f3ff07 starts running under Experiment 892219596593616876 ...
[84078:MainThread](2026-07-16 19:23:39,824) INFO - qlib.workflow - [record_temp.py:197] - Signal record 'pred.pkl' has been saved as the artifact of the Experiment 892219596593616876
[84078:MainThread](2026-07-16 19:23:39,849) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


'The following are prediction results of the LGBModel model.'
                          score
datetime   instrument          
2017-01-03 SH600000   -0.039979
           SH600008    0.001625
           SH600009    0.029651
           SH600010   -0.004148
           SH600015   -0.131706


[84078:MainThread](2026-07-16 19:24:12,402) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[84078:MainThread](2026-07-16 19:24:12,406) WARNING - qlib.online operator - [exchange.py:219] - $close field data contains nan.
[84078:MainThread](2026-07-16 19:24:12,412) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.
[84078:MainThread](2026-07-16 19:24:12,413) WARNING - qlib.online operator - [exchange.py:228] - trade unit 100 is not supported in adjusted_price mode.
[84078:MainThread](2026-07-16 19:24:22,827) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!
[84078:MainThread](2026-07-16 19:24:22,827) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md
[84078:MainThread](20

backtest loop:   0%|          | 0/871 [00:00<?, ?it/s]

/Users/xuxin/Documents/dev/qlib/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/xuxin/Documents/dev/qlib/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/xuxin/Documents/dev/qlib/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
[84078:MainThread](2026-07-16 19:24:28,029) INFO - qlib.workflow - [record_temp.py:520] - Portfolio analysis record 'port_analysis_1day.pkl' has been saved as the artifact of the Experiment 892219596593616876
[84078:MainThread](2026-07-16 19:24:28,034) INFO - qlib.workflow - [record_temp.py:545] - Indicator analysis record 'indicator_analysis_1day.pkl' has been saved as the artifact of the Experiment 892219596593616876


'The following are analysis results of benchmark return(1day).'
                       risk
mean               0.000477
std                0.012295
annualized_return  0.113561
information_ratio  0.598699
max_drawdown      -0.370479
'The following are analysis results of the excess return without cost(1day).'
                       risk
mean               0.000755
std                0.005767
annualized_return  0.179653
information_ratio  2.019134
max_drawdown      -0.070401
'The following are analysis results of the excess return with cost(1day).'
                       risk
mean               0.000560
std                0.005765
annualized_return  0.133179
information_ratio  1.497390
max_drawdown      -0.081073
'The following are analysis results of indicators(1day).'
     value
ffr    1.0
pa     0.0
pos    0.0


[84078:MainThread](2026-07-16 19:24:28,474) INFO - qlib.timer - [log.py:127] - Time cost: 0.009s | waiting `async_log` Done


# analyze graphs

In [10]:
!pip install statsmodels

In [11]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
print(recorder)
pred_df = recorder.load_object("pred.pkl")
report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

ModuleNotFoundError: No module named 'statsmodels'

## analysis position

### report

In [ ]:
analysis_position.report_graph(report_normal_df)

### risk analysis

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

## analysis model

In [ ]:
label_df = dataset.prepare("test", col_set="label")
label_df.columns = ["label"]

### score IC

In [ ]:
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
analysis_position.score_ic_graph(pred_label)

### model performance

In [ ]:
analysis_model.model_performance_graph(pred_label)